# Fase 1. Ingesta y auditoría inicial

Este notebook prepara la exploración inicial del conjunto de datos `BBDD_ML_TAREA.csv` siguiendo un criterio reproducible y orientado al modelado. El objetivo es identificar estructura, tipos, valores perdidos, duplicados, variables sospechosas por codificación y posibles redundancias antes de cualquier fase de entrenamiento.

La exploración se mantiene separada de la memoria en LaTeX para evitar mezclar análisis interactivo con resultados definitivos.

## 1. Preparación del entorno

En esta sección se importan las librerías básicas y se define la ruta relativa al fichero original.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 120)

DATA_PATH = Path('..') / 'data' / 'raw' / 'BBDD_ML_TAREA.csv'
RANDOM_STATE = 42

DATA_PATH.resolve()

WindowsPath('D:/Copias de seguridad/2025-2026/Master UCM/Módulo 11/Tarea/data/raw/BBDD_ML_TAREA.csv')

## 2. Carga e inspección estructural

Se carga el conjunto de datos tratando `NA` como valor perdido explícito y se revisan dimensiones, nombres de columnas y primeras filas.

In [2]:
df = pd.read_csv(DATA_PATH, na_values=['NA'])

display(df.head())
print('Dimensiones:', df.shape)
print('Columnas:', list(df.columns))
df.info()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,Y
0,43,121,415.0,1118,0,0,0,86.1,100,14.64,259.8,113,22.08,148.0,79.0,6.66,9.1,9,2.46,2,0
1,7,170,510.0,3326,0,0,0,184.1,106,31.30,204.9,70,17.42,224.3,133.0,10.09,9.8,3,2.65,2,0
2,31,96,510.0,2146,0,0,0,150.0,122,25.50,218.5,116,18.57,212.4,89.0,9.56,9.8,1,2.65,3,0
3,17,90,415.0,387,0,0,0,193.7,83,32.93,154.2,79,13.11,299.0,60.0,13.46,12.7,3,3.43,1,0
4,7,61,415.0,313,0,0,0,140.6,89,23.90,172.8,128,14.69,212.4,97.0,9.56,13.6,4,3.67,1,0


Dimensiones: (9200, 21)
Columnas: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'Y']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9200 entries, 0 to 9199
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      9200 non-null   int64  
 1   V2      9200 non-null   int64  
 2   V3      9125 non-null   float64
 3   V4      9200 non-null   int64  
 4   V5      9200 non-null   int64  
 5   V6      9200 non-null   int64  
 6   V7      9200 non-null   int64  
 7   V8      9200 non-null   float64
 8   V9      9200 non-null   int64  
 9   V10     9174 non-null   float64
 10  V11     9200 non-null   float64
 11  V12     9200 non-null   int64  
 12  V13     9200 non-null   float64
 13  V14     9128 non-null   float64
 14  V15     9140 non-null   float64
 15  V16     9200 non-null   float64
 16  V17     9200 non-null   float64
 17  V18     9200 n

## 3. Auditoría de calidad de datos

Se cuantifican valores perdidos, duplicados exactos, cardinalidad de columnas y distribución de la variable objetivo.

In [3]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df) * 100).round(2)
quality_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
    'n_unique': df.nunique(dropna=True),
    'dtype': df.dtypes.astype(str)
})

display(quality_summary)
print('Filas duplicadas exactas:', df.duplicated().sum())
print('Distribución de Y:')
display(df['Y'].value_counts(dropna=False).sort_index())
display(df['Y'].value_counts(normalize=True).sort_index().rename('proportion'))

,missing_count,missing_pct,n_unique,dtype
V1,0,0.00,51,int64
V10,26,0.28,1742,float64
V11,0,0.00,1646,float64
V12,0,0.00,118,int64
V13,0,0.00,1460,float64
V14,72,0.78,1622,float64
V15,60,0.65,124,float64
V16,0,0.00,951,float64
V17,0,0.00,166,float64
V18,0,0.00,21,int64


Filas duplicadas exactas: 5662
Distribución de Y:


Y
0    4600
1    4600
Name: count, dtype: int64

Y
0    0.5
1    0.5
Name: proportion, dtype: float64

## 4. Clasificación conceptual de variables

Aquí se fija una primera lectura semántica de las variables para distinguir identificadores, categóricas nominales, binarias y numéricas. Esta clasificación se usa después para diseñar el preprocesamiento sin introducir fuga de información.

In [ ]:
conceptual_groups = {
    'identificador_candidato': ['V4'],
    'nominales_codificadas': ['V1', 'V3'],
    'binarias': ['V5', 'V6', 'Y'],
    'discretas': ['V2', 'V7', 'V9', 'V12', 'V15', 'V18', 'V20'],
    'continuas': ['V8', 'V10', 'V11', 'V13', 'V14', 'V16', 'V17', 'V19']
}

for name, cols in conceptual_groups.items():
    present_cols = [col for col in cols if col in df.columns]
    print(f'{name}: {present_cols}')

## 5. Revisión de posibles anomalías y baja variabilidad

Se revisan columnas constantes, rangos básicos y patrones de ceros o valores extremos que puedan requerir tratamiento específico en el informe.

In [ ]:
low_variability = df.nunique(dropna=False).sort_values()
display(low_variability)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
display(df[numeric_cols].describe().T)

## 6. Redundancia entre variables de uso y coste

Las parejas minuto-coste se analizan de forma preliminar para verificar la redundancia conceptual entre bloques de llamadas. La decisión de conservar o reducir variables se tomará con evidencia empírica y no por presunción.

In [ ]:
redundancy_pairs = [('V8', 'V10'), ('V11', 'V13'), ('V14', 'V16'), ('V17', 'V19')]
existing_pairs = [(a, b) for a, b in redundancy_pairs if a in df.columns and b in df.columns]

corr_rows = []
for a, b in existing_pairs:
    corr_rows.append({
        'pair': f'{a}-{b}',
        'pearson_corr': df[[a, b]].corr().iloc[0, 1] if df[[a, b]].dropna().shape[0] > 1 else np.nan
    })

redundancy_summary = pd.DataFrame(corr_rows)
display(redundancy_summary)

## 7. Próximos pasos

Una vez validada esta auditoría básica, el notebook se ampliará con: imputación específica, análisis de distribuciones, estudio de correlaciones, tratamiento de duplicados y preparación del conjunto de datos para los modelos del resto de la práctica.

## 8. Diagnóstico de variable identificadora y duplicados

Se revisa si \(V4\) se comporta como identificador (alta cardinalidad) y se mide el impacto de duplicados exactos y duplicados ignorando \(V4\). Este análisis permite justificar si \(V4\) debe excluirse del modelado.

In [ ]:
id_check = pd.DataFrame({
    'n_unique': df.nunique(dropna=False),
    'unique_ratio': (df.nunique(dropna=False) / len(df)).round(4)
}).sort_values('unique_ratio', ascending=False)

display(id_check.head(10))

exact_duplicates = int(df.duplicated().sum())
duplicates_wo_v4 = int(df.drop(columns=['V4']).duplicated().sum()) if 'V4' in df.columns else np.nan

print('Duplicados exactos (todas las columnas):', exact_duplicates)
print('Duplicados ignorando V4:', duplicates_wo_v4)

## 9. Distribución de la variable objetivo

Se incluye una visualización simple para verificar desbalance de clases en \(Y\).

In [ ]:
target_counts = df['Y'].value_counts(dropna=False).sort_index()
target_pct = (target_counts / target_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(target_counts.index.astype(str), target_counts.values)
ax.set_title('Distribucion de la variable objetivo Y')
ax.set_xlabel('Clase')
ax.set_ylabel('Frecuencia')
for i, (count, pct) in enumerate(zip(target_counts.values, target_pct.values)):
    ax.text(i, count, f'{count} ({pct}%)', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 10. Exportación de tablas de auditoría

Para mantener trazabilidad entre exploración y memoria, se exportan las tablas clave de esta fase a la carpeta de salidas del proyecto.

In [ ]:
outputs_tables = Path('..') / 'outputs' / 'tables'
outputs_tables.mkdir(parents=True, exist_ok=True)

quality_summary.to_csv(outputs_tables / 'fase1_quality_summary.csv', index=True)
id_check.to_csv(outputs_tables / 'fase1_id_check.csv', index=True)
redundancy_summary.to_csv(outputs_tables / 'fase1_redundancy_summary.csv', index=False)

target_table = pd.DataFrame({
    'count': target_counts,
    'percentage': target_pct
})
target_table.to_csv(outputs_tables / 'fase1_target_distribution.csv', index=True)

print('Tablas exportadas en:', outputs_tables.resolve())

## 11. Checklist de cierre de Fase 1

- Estructura de datos validada (dimensiones, tipos, nombres).
- Valores perdidos cuantificados y localizados.
- Duplicados medidos (exactos y sin variable candidata a identificador).
- Variable objetivo analizada en frecuencia y porcentaje.
- Clasificación conceptual de variables documentada.
- Redundancia minuto-coste evaluada de forma preliminar.

Con este bloque queda preparada la base para diseñar el protocolo de particionado y el preprocesamiento específico por técnica en la siguiente fase.